# Creación de un prototipo de asistente virtual

Esta práctica propone la creación de un prototipo del sistema de traducción que permite recibir órdenes en un idioma y ejecutarlos en otro idioma. Por ejemplo, el usuario podría decir "open the door" y el sistema respondería "abre la puerta". Este prototipo se basará en el uso de modelos de HugginFace previamente entrenados para la traducción y la síntesis de voz.

## Buscar modelos previamente entrenados

El primer paso es buscar modelos previamente entrenados para las tareas solicitadas. Investigue Huggingface para seleccionar modelos que mejor satisfagan las necesidades del prototipo.

Cuando haya seleccionado los modelos, modifique las siguientes variables para incluirlas en el prototipo:

In [1]:
# Modelo de voz a texto (OpenAI Whisper es el estándar de oro)
MODELO_VOZ_A_TEXTO = "openai/whisper-tiny"

# Modelo de traducción (Helsinki-NLP son los más ligeros y fiables)
MODELO_TRADUCCION = "Helsinki-NLP/opus-mt-en-es"

# Modelo de texto a voz (Sunao es moderno, pero para algo sencillo usaremos Facebook)
MODELO_TEXTO_A_VOZ = "facebook/mms-tts-spa"

## Implementación del prototipo

Una vez que se seleccionan los modelos previamente entrenados, implementa el prototipo.Para hacer esto, puede seguir los siguientes pasos:

1. Cree una función que, dada una orden de voz, lo transforme en texto (_Automatic Speech Recognition_ ASR). Para hacer esto, puede usar el modelo `MODELO_VOZ_A_TEXTO` previamente seleccionado. Como ejemplo, puede usar el siguiente archivo de voz: [OpenTheDoor.wav](OpenTheDoor.wav).

In [3]:
# Creamos la tubería para la conversión de voz a texto
from transformers import pipeline

# Creamos la tubería para la conversión de voz a texto
asr_pipe = pipeline("automatic-speech-recognition", model=MODELO_VOZ_A_TEXTO)

# Convertimos la voz en texto
text_en = asr_pipe("OpenTheDoor.wav")["text"]
print(f"Texto detectado (EN): {text_en}")


Device set to use cpu
`return_token_timestamps` is deprecated for WhisperFeatureExtractor and will be removed in Transformers v5. Use `return_attention_mask` instead, as the number of frames can be inferred from it.
Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


Texto detectado (EN):  Open the door.


2. Crea una función que, dado un texto y un idioma, lo traduce al español (_machine translation_). Para hacer esto, puede usar el modelo `MODELO_TRADUCCION` seleccionado anteriormente. Como ejemplo, puede usar el siguiente pedido en texto: "Abra la puerta".

In [4]:
# Creamos la tubería para la traducción
translator_pipe = pipeline("translation", model=MODELO_TRADUCCION)

# Traducimos el texto al español
text_es = translator_pipe(text_en)[0]["translation_text"]
print(f"Texto traducido (ES): {text_es}")

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/802k [00:04<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")
Device set to use cpu


Texto traducido (ES): Abre la puerta.


3. Crea una funció que, donat un text, el sintetitzi en veu (_Text to speech_). Per fer-ho pots utilitzar el model `MODEL_TEXT_A_VEU` seleccionat anteriorment. Com a exemple pots utilitzar el text "abre la puerta".

In [5]:
# Creamos la tubería para la síntesis de voz
tts_pipe = pipeline("text-to-speech", model=MODELO_TEXTO_A_VOZ)

# Sintetizamos el texto en voz
out = tts_pipe(text_es)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/145M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/497 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/47.0 [00:00<?, ?B/s]

Device set to use cpu


In [6]:
# mostramos el texto sintetizado como un reproductor

from IPython.display import Audio

Audio(out["audio"], rate=out['sampling_rate'])

4.- Une las tres funciones anteriores en una sola función que, dada una voz y un idioma, lo transforma en texto, lo traduce en otro idioma y la sintetizando en la voz. Como ejemplo, puede usar el siguiente audio de voz: [OpenTheDoor.wav](OpenTheDoor.wav).

In [7]:
from IPython.display import Audio
import torch

def assistant(voice_order):
    # 1. Convertimos la voz a texto (ASR)
    # Usamos las tuberías ya creadas arriba
    asr_result = asr_pipe(voice_order)
    texto_ingles = asr_result["text"]

    # 2. Traducimos el texto al castellano
    traduccion = translator_pipe(texto_ingles)
    texto_castellano = traduccion[0]["translation_text"]

    print(f"El usuario dijo: '{texto_ingles}'")
    print(f"Traducción: '{texto_castellano}'")

    # 3. Sintetizamos el texto en voz (TTS)
    # Devolvemos el diccionario con el audio y el sampling_rate
    voz_out = tts_pipe(texto_castellano)

    return voz_out

# Ejecutamos el asistente
out = assistant("OpenTheDoor.wav")

# Reproducimos el resultado
Audio(out["audio"], rate=out['sampling_rate'])

El usuario dijo: ' Open the door.'
Traducción: 'Abre la puerta.'
